# 01 · Vector search: the baseline and its gap

Look at the data, then run the simplest retrieval there is, and watch where it fails.

| | |
| --- | --- |
| **Needs** | `00_setup.ipynb` in the same folder. The first code cell runs it for you. |
| **LLM** | Optional. With `RUN_LLM = False` (the default in `00_setup.ipynb`) you still see everything retrieval does. |
| **Previous / next** | [`00_setup.ipynb`](00_setup.ipynb) / [`02_bm25_ner.ipynb`](02_bm25_ner.ipynb) |

*The code mirrors the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval) in `instaclustr/labs`. Lines marked "Mirrors" name the file in the journey that each piece comes from. This notebook only searches: it never writes to the cluster.*

### Run the setup

The next cell runs `00_setup.ipynb`. It connects to the cluster, loads the two small models, and defines the helper functions used below. You should see `[PASS]` lines. If you see `[FAIL]`, fix that before going on, using the notes in `00_setup.ipynb`.

In [ ]:
%run ./00_setup.ipynb

## The data

The index holds **260 recipes**. Each has its text, its source website, and metadata, including `cautions` (allergen warnings such as Sulfites) and a `tier`. The tier is a governance label that notebook 05 explains: recipes with any caution are `hot`, the rest are `lt` (long-term).

The cell below asks the cluster to count them. It is an aggregation, which is a read-only search.

In [ ]:
body = {
    "size": 0, "track_total_hits": True,
    "aggs": {
        "by_tier": {"terms": {"field": "tier", "size": 5}},
        "sulfites": {"filter": {"term": {"cautions": "sulfites"}}},
        "top_cautions": {"terms": {"field": "cautions", "size": 6}},
    },
}
r = search(IDX["vector"], body)
print("recipes:", r["hits"]["total"]["value"])
print("by tier:", {b["key"]: b["doc_count"] for b in r["aggregations"]["by_tier"]["buckets"]})
print("with a Sulfites caution:", r["aggregations"]["sulfites"]["doc_count"])
print("most common cautions:", {b["key"]: b["doc_count"] for b in r["aggregations"]["top_cautions"]["buckets"]})

**Expected:** 260 recipes, about 201 with a Sulfites caution, and a tier split of roughly 209 `hot` and 51 `lt`.

### The two "Fish and Chips" recipes

The rest of the notebook turns on two recipes that share a name but come from different websites. They differ in one safety-relevant fact: one lists a Sulfites caution and the other does not. A good retriever must tell them apart.

In [ ]:
body = {"size": 10, "_source": RECIPE_FIELDS,
        "query": {"match_phrase": {"recipe_name": "fish and chips"}}}
pair = normalize_hits(search(IDX["vector"], body))
show_hits(pair)

**Expected:** the two recipes named Fish and Chips, one from `thegratefulgirlcooks.com` with **no** cautions (tier `lt`), and one from `womensweeklyfood.com.au` with a **Sulfites** caution (tier `hot`). Other fish-and-chips-style recipes may also appear.

Keep these two in mind. Every step below asks the same two questions:

- `Q_GRATEFUL`: *Does the thegratefulgirlcooks.com Fish and Chips recipe contain sulfites?* (Answer: no.)
- `Q_WOMENS`: *Does the womensweeklyfood.com.au Fish and Chips recipe contain sulfites?* (Answer: yes.)

## Baseline vector RAG

**The idea.** Turn the question into a vector and find the recipes whose vectors are nearest. This is *semantic* search: it matches on meaning, not on exact words.

**The failure to look for.** Both questions mean almost the same thing and mention the same dish, so their vectors are almost identical. The recipe's *website name* barely moves the vector. Watch whether the recipe each question names actually shows up.

*Mirrors:* `scripts/baseline-vector-rag/src/query.py`.

In [ ]:
def vector_search(question, top_k=5, candidates=25, index=None):
    body = {"size": top_k, "_source": RECIPE_FIELDS,
            "query": {"knn": {"embedding": {"vector": embed(question), "k": candidates}}}}
    return normalize_hits(search(index or IDX["vector_baseline"], body))


def ask_vector(question, top_k=5):
    hits = vector_search(question, top_k)
    prompt = f"Question:\n{question}\n\nContext:\n{build_context(hits) or 'No context available.'}"
    return answer(prompt, VECTOR_SYSTEM), hits


for question, domain in [(Q_GRATEFUL, "thegratefulgirlcooks"), (Q_WOMENS, "womensweeklyfood")]:
    reply, hits = ask_vector(question)
    print("\nQ:", question)
    show_hits(hits)
    print(f"  -> the named recipe ({domain}) was retrieved: {has_source(hits, domain)}")
    print("  ANSWER:", reply)

**Expected:** 5 hits per question. For the first question, the `thegratefulgirlcooks.com` recipe is **not** retrieved, so the system cannot answer about the right recipe. For the second, the `womensweeklyfood.com.au` recipe is retrieved. Scores and answer wording can vary. This is the **retrieval gap**: vector search found *similar* recipes, not the *named* one.

**Try this:** change `top_k` to 10 and rerun. Does the missing recipe appear?

### Checkpoint

You should be able to say why vector search did not retrieve the `thegratefulgirlcooks.com` recipe for a question that names it. (Both questions mean almost the same thing, so their vectors are almost identical, and the website name barely moves a vector.)

Next: open [`02_bm25_ner.ipynb`](02_bm25_ner.ipynb).